# Chess Style Model — Colab Training (T4)

Bumped-capacity CNN that imitates your Chess.com move choices (behavior cloning). This is Phase 2 of the `chess` repo's build plan — run top to bottom.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Upload your `<username>_positions.jsonl` (produced locally by `build_dataset.py`) when prompted.
3. Optionally upload an existing `<username>_policy.pt` checkpoint to resume instead of training from scratch.
4. At the end, download the trained checkpoint and drop it into `models/` in the local `chess` repo — `predict.py` and the Phase 3 blend engine pick it up automatically.

The `PolicyNet` architecture here is mirrored in the local `train_style.py` so checkpoints are interchangeable between laptop and Colab.

In [ ]:
!pip install -q python-chess

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (switch to T4 in Runtime settings)")

## 1. Upload dataset

In [ ]:
from google.colab import files

print("Upload your <username>_positions.jsonl file:")
uploaded = files.upload()
dataset_path = list(uploaded.keys())[0]
username = dataset_path.replace("_positions.jsonl", "").replace(".jsonl", "")
print(f"Dataset: {dataset_path}  |  username inferred: {username}")

## 2. (Optional) Resume from an existing checkpoint
Set `RESUME_FROM_CHECKPOINT = True` below if you have a `<username>_policy.pt` to continue training (either from a previous Colab run or from the local laptop script).

In [ ]:
RESUME_FROM_CHECKPOINT = False  #@param {type:"boolean"}

ckpt_path = None
if RESUME_FROM_CHECKPOINT:
    print(f"Upload your {username}_policy.pt checkpoint:")
    ckpt_uploaded = files.upload()
    ckpt_path = list(ckpt_uploaded.keys())[0]
    print(f"Will resume from: {ckpt_path}")
else:
    print("Training from scratch.")

## 3. Model + encoding
Same board/move encoding as the local `train_style.py`. Capacity bumped (deeper, wider, BatchNorm + Dropout) — the laptop version couldn't afford this, T4 can.

In [ ]:
import json
import chess
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- board encoding: 12 piece planes x 8x8 ---
PIECE_TO_PLANE = {
    (chess.PAWN, True): 0, (chess.KNIGHT, True): 1, (chess.BISHOP, True): 2,
    (chess.ROOK, True): 3, (chess.QUEEN, True): 4, (chess.KING, True): 5,
    (chess.PAWN, False): 6, (chess.KNIGHT, False): 7, (chess.BISHOP, False): 8,
    (chess.ROOK, False): 9, (chess.QUEEN, False): 10, (chess.KING, False): 11,
}


def encode_board(fen: str) -> torch.Tensor:
    board = chess.Board(fen)
    x = torch.zeros(12, 8, 8)
    for sq, piece in board.piece_map().items():
        plane = PIECE_TO_PLANE[(piece.piece_type, piece.color)]
        r, c = divmod(sq, 8)
        x[plane, r, c] = 1.0
    return x


# --- move encoding: from_square(64) x to_square(64) = 4096 classes (ignores underpromotion choice) ---
def encode_move(uci: str) -> int:
    move = chess.Move.from_uci(uci)
    return move.from_square * 64 + move.to_square


def decode_move_index(idx: int) -> chess.Move:
    return chess.Move(idx // 64, idx % 64)


class PolicyNet(nn.Module):
    """Bumped-capacity version — mirror any change here in local train_style.py."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(12, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 1024), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(1024, 4096),
        )

    def forward(self, x):
        return self.head(self.conv(x))

## 4. Load dataset (train/val split)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

def load_dataset(jsonl_path: str):
    xs, ys = [], []
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            xs.append(encode_board(row["fen"]))
            ys.append(encode_move(row["move"]))
    return torch.stack(xs), torch.tensor(ys)

X, y = load_dataset(dataset_path)
print(f"Loaded {len(X)} of your moves.")

full_ds = TensorDataset(X, y)
val_size = max(1, int(0.1 * len(full_ds)))
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size])

BATCH_SIZE = 128  #@param {type:"integer"}
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
print(f"Train: {train_size}  Val: {val_size}")

## 5. Train

In [ ]:
EPOCHS = 60  #@param {type:"integer"}
LR = 1e-3  #@param {type:"number"}

model = PolicyNet().to(device)
if ckpt_path is not None:
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"Resumed from: {ckpt_path}")

opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()
        train_loss += loss.item() * len(xb)
        train_correct += (logits.argmax(1) == yb).sum().item()
        train_total += len(xb)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            val_correct += (logits.argmax(1) == yb).sum().item()
            val_total += len(xb)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"epoch {epoch+1}/{EPOCHS}  loss={train_loss/train_total:.4f}  "
              f"train_acc={train_correct/train_total:.3f}  val_acc={val_correct/max(1,val_total):.3f}")

## 6. Save + download checkpoint
Drop the downloaded file into `models/` in the local `chess` repo (same filename).

In [ ]:
out_name = f"{username}_policy.pt"
torch.save(model.state_dict(), out_name)
print(f"Saved -> {out_name}")
files.download(out_name)